In [2]:
"Write program to implement multi-layer perceptron with back propagation"

'Write program to implement multi-layer perceptron with back propagation'

In [1]:
import numpy as np

class MLP:
    """Fully connected MLP for binary classification, trained with backpropagation."""

    def __init__(self, layer_sizes, seed=0):
        if len(layer_sizes) < 2:
            raise ValueError("Provide at least an input layer and an output layer.")
        if layer_sizes[-1] != 1:
            raise ValueError("This implementation expects one output neuron.")

        rng = np.random.default_rng(seed)
        self.weights = []
        self.biases = []

        for n_in, n_out in zip(layer_sizes[:-1], layer_sizes[1:]):
            scale = np.sqrt(2.0 / (n_in + n_out))
            self.weights.append(rng.normal(0, scale, (n_in, n_out)))
            self.biases.append(np.zeros((1, n_out)))

    @staticmethod
    def _sigmoid(z):
        z = np.clip(z, -500, 500)
        return 1.0 / (1.0 + np.exp(-z))

    def predict_proba(self, X):
        activations = np.asarray(X, dtype=float)
        for weight, bias in zip(self.weights, self.biases):
            activations = self._sigmoid(activations @ weight + bias)
        return activations

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

    def fit(self, X, y, epochs=10_000, learning_rate=1.0):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float).reshape(-1, 1)

        if X.ndim != 2 or X.shape[1] != self.weights[0].shape[0]:
            raise ValueError("X must have shape (n_samples, input_size).")
        if len(X) != len(y):
            raise ValueError("X and y must contain the same number of samples.")

        n_samples = len(X)

        for _ in range(epochs):
            # Forward pass
            activations = [X]
            for weight, bias in zip(self.weights, self.biases):
                activations.append(
                    self._sigmoid(activations[-1] @ weight + bias)
                )

            # Backward pass: sigmoid output with binary cross-entropy
            delta = activations[-1] - y
            grad_weights = [None] * len(self.weights)
            grad_biases = [None] * len(self.biases)

            for layer in reversed(range(len(self.weights))):
                grad_weights[layer] = (
                    activations[layer].T @ delta / n_samples
                )
                grad_biases[layer] = np.mean(delta, axis=0, keepdims=True)

                if layer > 0:
                    hidden = activations[layer]
                    delta = (delta @ self.weights[layer].T) * hidden * (1 - hidden)

            for layer in range(len(self.weights)):
                self.weights[layer] -= learning_rate * grad_weights[layer]
                self.biases[layer] -= learning_rate * grad_biases[layer]


# XOR example
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y = np.array([0, 1, 1, 0])

model = MLP([2, 8, 1], seed=1)
model.fit(X, y, epochs=10_000, learning_rate=1.0)

print("Probabilities:", model.predict_proba(X).ravel())
print("Predictions:  ", model.predict(X).ravel())

Probabilities: [5.54527493e-04 9.99493355e-01 9.99535447e-01 4.20110186e-04]
Predictions:   [0 1 1 0]
